In [3]:
import numpy as np

fname = "results_pz_0.0_Lx_16_Nd_1000_NT_7680.npz"
data = np.load(fname)

# どんなデータが入っているか（キーの一覧）
print("keys:", data.files)

# 各キーの中身の形状や型を確認
for k in data.files:
    arr = data[k]
    print(f"{k}: shape={arr.shape}, dtype={arr.dtype}")


keys: ['pg', 'TEE_ave', 'TEE_err', 'TEE_var', 'R2x_ave', 'R2x_err', 'R2x_var', 'R2z_ave', 'R2z_err', 'R2z_var', 'R2x_loop_ave', 'R2x_loop_err', 'R2x_loop_var', 'R2z_loop_ave', 'R2z_loop_err', 'R2z_loop_var', 'dep_x_ave', 'dep_x_err', 'dep_x_var', 'dep_z_ave', 'dep_z_err', 'dep_z_var']
pg: shape=(21,), dtype=float64
TEE_ave: shape=(21,), dtype=float64
TEE_err: shape=(21,), dtype=float64
TEE_var: shape=(21,), dtype=float64
R2x_ave: shape=(21,), dtype=float64
R2x_err: shape=(21,), dtype=float64
R2x_var: shape=(21,), dtype=float64
R2z_ave: shape=(21,), dtype=float64
R2z_err: shape=(21,), dtype=float64
R2z_var: shape=(21,), dtype=float64
R2x_loop_ave: shape=(21,), dtype=float64
R2x_loop_err: shape=(21,), dtype=float64
R2x_loop_var: shape=(21,), dtype=float64
R2z_loop_ave: shape=(21,), dtype=float64
R2z_loop_err: shape=(21,), dtype=float64
R2z_loop_var: shape=(21,), dtype=float64
dep_x_ave: shape=(21,), dtype=float64
dep_x_err: shape=(21,), dtype=float64
dep_x_var: shape=(21,), dtype=float64

ブートストラップで分散の誤差を作成："results_px_*_Lx_*_Nd_*_NT_*.npz"→"results_px_*_Lx_*_Nd_*_NT_*.npz"

In [4]:
import numpy as np
import glob, re, os
from numpy.random import default_rng

B = 1000                  # ブートストラップ反復回数（200–2000程度で調整）
rng = default_rng(123)    # 乱数種

# N_eff の定義: "Nd"（推奨, 保守的） or "NdNT"（独立なら）
N_mode = "Nd"

files = glob.glob("results_pz_*_Lx_*_Nd_*_NT_*.npz")

for fn in files:
    m = re.search(r"pz_([0-9.]+)_Lx_(\d+)_Nd_(\d+)_NT_(\d+)\.npz$", fn)
    if not m:
        print(f"[skip] name pattern mismatch: {fn}")
        continue
    Nd = int(m.group(3)); NT = int(m.group(4))
    N_eff = Nd if N_mode == "Nd" else Nd*NT

    data = np.load(fn)
    if "R2x_loop_var" not in data.files:
        print(f"[skip] no R2x_loop_var in {fn}")
        continue

    s2 = np.asarray(data["R2x_var"], float)  # 形状: (nPg,)
    s2 = np.clip(s2, 0.0, None)
    df = max(N_eff - 1, 1)

    # χ²ブートストラップ：sigma2* を χ² から生成し、対応する s2* を作る
    # 由来: (df * s2) / sigma2 ~ χ²_df  ⇒  sigma2 = df * s2 / χ²
    # 観測 s2 を真値の近似に使い、sigma2* を生成 → そこから s2* を再構成しても良いが、
    # 実務的には「s2* = df * s2 / χ²(df)」で十分に近い分布を再現できます。
    # （s2 が真の sigma2 を近似しているとみなす近似）
    chi2_samples = rng.chisquare(df, size=(B, s2.size))
    s2_boot = (df * s2[None, :]) / chi2_samples  # 形状 (B, nPg)

    # ブートストラップ標準偏差を誤差として採用
    s2_err = s2_boot.std(axis=0, ddof=1)

    # 保存（別名ファイル）
    out = {k: data[k] for k in data.files}
    out["R2x_loop_var_err"] = s2_err.astype(float)
    out["R2x_loop_var_Neff"] = np.array(N_eff, dtype=int)

    base, ext = os.path.splitext(fn)
    out_fn = base + "_with_err" + ext
    np.savez(out_fn, **out)
    print(f"[ok] wrote {out_fn} (N_eff={N_eff}, B={B})")


[ok] wrote results_pz_0.0_Lx_16_Nd_1000_NT_7680_with_err.npz (N_eff=1000, B=1000)
[ok] wrote results_pz_0.0_Lx_14_Nd_1000_NT_5880_with_err.npz (N_eff=1000, B=1000)
[ok] wrote results_pz_0.0_Lx_12_Nd_1000_NT_4320_with_err.npz (N_eff=1000, B=1000)
[ok] wrote results_pz_0.0_Lx_6_Nd_1000_NT_1080_with_err.npz (N_eff=1000, B=1000)
[ok] wrote results_pz_0.0_Lx_20_Nd_500_NT_12000_with_err.npz (N_eff=500, B=1000)
[ok] wrote results_pz_0.0_Lx_10_Nd_1000_NT_3000_with_err.npz (N_eff=1000, B=1000)
[ok] wrote results_pz_0.0_Lx_8_Nd_1000_NT_1920_with_err.npz (N_eff=1000, B=1000)
[ok] wrote results_pz_0.0_Lx_18_Nd_1000_NT_9720_with_err.npz (N_eff=1000, B=1000)
